# Test: Simplexity HMM generation (PyTorch)

**Requires Python ≥ 3.12** (simplexity does not support 3.9). Use a 3.12+ env, then from repo root: `pip install -r requirements.txt` and `pip install -r requirements-simplexity.txt`.

This notebook uses [simplexity](https://github.com/Astera-org/simplexity), which **supports multiple frameworks including PyTorch** ([multi-framework architecture](https://deepwiki.com/Astera-org/simplexity)). We use simplexity for HMM generation and its **JAX–PyTorch conversion** so all data is in **PyTorch tensors** for the rest of the project (belief state geometry, Mess3, LLM prompting).

**Simplexity** provides:
- `build_hidden_markov_model(process_name, process_params)` — build HMMs (e.g. `"mess3"`, `"coin"`, `"even_ones"`)
- `generate_data_batch` / `generate_data_batch_with_full_history` — generate sequences and belief-state history
- `simplexity.utils.pytorch_utils.jax_to_torch` — zero-copy conversion to PyTorch (DLPack when on GPU)

In [25]:
import simplexity; print(simplexity.__file__)

/home/chiragr2/miniconda3/envs/py312/lib/python3.12/site-packages/simplexity/__init__.py


In [26]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.insert(0, os.path.abspath('../..'))

import jax
import jax.numpy as jnp
import numpy as np
import torch
import matplotlib.pyplot as plt

# Simplexity's official JAX→PyTorch conversion (zero-copy via DLPack on GPU)
from simplexity.utils.pytorch_utils import jax_to_torch, torch_to_jax

from src.hmm.hmm import Mess3HMM, belief_to_barycentric

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Build an HMM (Mess3)

Mess3 is a 3-state, 3-symbol HMM used in computational mechanics. Parameters:
- `x`: controls transition structure
- `a`: asymmetry (in simplexity the param is `a`; in my `Mess3HMM` it was `alpha`)

We use `x=0.05`, `a=0.85` to stay close to the project's existing Mess3 setup.

Find more details in **simplexity/generative_processes/builder.py**

In [ ]:
from simplexity.generative_processes.builder import build_hidden_markov_model

# hmm = build_hidden_markov_model(
#     "mess3",
#     process_params={"x": 0.05, "a": 0.85},
#     device=None,
# )

hmm = build_hidden_markov_model(
    "leopard",
    process_params={"x": 0.5},
    device=None,
)

print(f"Vocab size (symbols): {hmm.vocab_size}")
print(f"Num hidden states: {hmm.num_states}")
print(f"Initial state (belief): {jax_to_torch(hmm.initial_state).tolist()}")

Vocab size (symbols): 2
Num hidden states: 3
Initial state (belief): [0.20641230046749115, 0.32916098833084106, 0.4644266963005066]


In [11]:
jax_to_torch(hmm.initial_state)

tensor([0.3333, 0.3333, 0.3333])

## 2. Generate sequences

Use `generate_data_batch` to sample sequences. It returns `(gen_states, inputs, labels)` where `inputs`/`labels` are consecutive tokens (for next-token prediction).

In [12]:
from simplexity.generative_processes.generator import generate_data_batch

batch_size = 20
sequence_len = 2000
key = jax.random.PRNGKey(42)
gen_states = jnp.tile(hmm.initial_state, (batch_size, 1))

_, inputs_jax, labels_jax = generate_data_batch(gen_states, hmm, batch_size, sequence_len, key)
# Convert to PyTorch for use in the rest of the project
inputs = jax_to_torch(inputs_jax)
labels = jax_to_torch(labels_jax)

print("inputs shape:", inputs.shape)   # (batch_size, sequence_len)
print("labels shape:", labels.shape)   # (batch_size, sequence_len)
print("Sample sequence (first batch, first 20 tokens):", inputs[0, :20].tolist())
print("Sample labels (first batch, first 20 tokens):", labels[0, :20].tolist())
print("Token map: 0=A, 1=B, 2=C for Mess3")

inputs shape: torch.Size([20, 1999])
labels shape: torch.Size([20, 1999])
Sample sequence (first batch, first 20 tokens): [1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1]
Sample labels (first batch, first 20 tokens): [1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1]
Token map: 0=A, 1=B, 2=C for Mess3


## 3. Generate with full belief-state history

Use `generate_data_batch_with_full_history` to get per-token belief states (for probing / visualization).

In [13]:
from simplexity.generative_processes.generator import generate_data_batch_with_full_history

key2 = jax.random.PRNGKey(42)
gen_states2 = jnp.tile(hmm.initial_state, (batch_size, 1)) # (batch_size, num_states) just the initial state repeated

# next_states, 
gen_result = generate_data_batch_with_full_history(
    gen_states2, hmm, batch_size, sequence_len, key2
)

# belief_states_jax, prefix_probs, inputs_h_jax, labels_h_jax 

# Convert to PyTorch
inputs_h_jax = gen_result["inputs"]
belief_states_jax = gen_result["belief_states"]
labels_h_jax = gen_result["labels"]
prefix_probs = gen_result["prefix_probabilities"]

belief_states = jax_to_torch(belief_states_jax)
inputs_h = jax_to_torch(inputs_h_jax)
labels_h = jax_to_torch(labels_h_jax)

print("belief_states shape:", belief_states.shape)  # (batch_size, sequence_len, num_states)
print("Belief state at t=0 (first batch):", belief_states[0, 0].tolist())
print("Belief state at t=5 (first batch):", belief_states[0, 5].tolist())

belief_states shape: torch.Size([20, 1999, 3])
Belief state at t=0 (first batch): [0.23531357944011688, 0.3333333730697632, 0.43135306239128113]
Belief state at t=5 (first batch): [0.5386343598365784, 0.16260863840579987, 0.29875698685646057]


In [14]:
belief_states.shape

torch.Size([20, 1999, 3])

In [15]:
belief_to_barycentric(belief_states.reshape(-1,3).cpu(), belief_states.reshape(-1,3).cpu())

In [16]:
hmm = build_hidden_markov_model(
    "mess3",
    process_params={"x": 0.05, "a": 0.85},
    device=None,
)

In [17]:
from simplexity.generative_processes.hidden_markov_model import HiddenMarkovModel
from simplexity.generative_processes.mixed_state_presentation import (
    LogMixedStateTreeGenerator,
    MixedStateTreeGenerator,
)

def _extract_hmm_msp(
    hmm: HiddenMarkovModel, max_seq_len: int
) -> tuple[np.ndarray, np.ndarray]:
    gen = LogMixedStateTreeGenerator(hmm, max_sequence_length=max_seq_len)
    tree = gen.generate()
    vals = list(tree.nodes.values())
    log_bs = jnp.array([v.log_belief_state for v in vals])
    log_probs = jnp.array([v.log_probability for v in vals])
    return np.array(jnp.exp(log_bs)), np.exp(np.array(log_probs))

In [18]:
gen = LogMixedStateTreeGenerator(hmm, max_sequence_length=6)
tree = gen.generate()
vals = list(tree.nodes.values())
log_bs = jnp.array([v.log_belief_state for v in vals])
log_probs = jnp.array([v.log_probability for v in vals])

In [19]:
sequences = list(tree.nodes.keys())  

In [20]:
belief_states, obs_prob = _extract_hmm_msp(hmm, 8)

In [21]:
belief_to_barycentric(belief_states.reshape(-1,3), belief_states.reshape(-1,3))

## 4. Observation distribution and sequence probability

Compute P(observation | belief state) and P(sequence) under the HMM.

In [45]:
# Observation distribution from initial state (simplexity uses JAX; convert to PyTorch)
obs_probs_jax = hmm.observation_probability_distribution(hmm.initial_state)
obs_probs = jax_to_torch(obs_probs_jax)
print("P(symbol | initial state):", obs_probs.tolist())

# Log-probability of one generated sequence (pass JAX array to simplexity)
seq_jax = jnp.array(inputs[0].numpy())
log_prob_jax = hmm.log_probability(seq_jax)
log_prob = float(log_prob_jax)
print(f"Log P(sequence): {log_prob:.4f}")

P(symbol | initial state): [0.4949999749660492, 0.5049999356269836]
Log P(sequence): -682.9632


In [46]:
obs_probs_all_jax = jnp.einsum('bns,vst->bnv', belief_states_jax, hmm.transition_matrices)

print("belief_states_jax shape:", belief_states_jax.shape)
print("obs_probs_all_jax shape:", obs_probs_all_jax.shape)

# Verify against single-state API
single_check = hmm.observation_probability_distribution(belief_states_jax[0, 1])
print("\nVerification — single-state API vs einsum:")
print("  single-state API:", single_check)
print("  einsum [0,0]:    ", obs_probs_all_jax[0, 1])
print("  match:", jnp.allclose(single_check, obs_probs_all_jax[0, 1]))

belief_states_jax shape: (20, 999, 3)
obs_probs_all_jax shape: (20, 999, 2)

Verification — single-state API vs einsum:
  single-state API: [0.5245307  0.47546932]
  einsum [0,0]:     [0.5245307 0.4754693]
  match: True


## 5. Other HMMs: coin, even_ones

Simplexity includes several built-in processes; we convert outputs to PyTorch.

In [47]:
# Coin process: P(head)=p, vocab size 2
coin_hmm = build_hidden_markov_model("coin", process_params={"p": 0.7})
print("Coin HMM vocab_size:", coin_hmm.vocab_size, "num_states:", coin_hmm.num_states)
gs = jnp.tile(coin_hmm.initial_state, (2, 1))
_, inp_jax, _ = generate_data_batch(gs, coin_hmm, 2, 20, jax.random.PRNGKey(0))
inp = jax_to_torch(inp_jax)
print("Coin sample sequence (PyTorch):", inp[0].tolist())

# Even ones: 2 states, 2 symbols
eo_hmm = build_hidden_markov_model("even_ones", process_params={"p": 0.5})
print("\nEven-ones HMM vocab_size:", eo_hmm.vocab_size, "num_states:", eo_hmm.num_states)

Coin HMM vocab_size: 2 num_states: 1
Coin sample sequence (PyTorch): [1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0]

Even-ones HMM vocab_size: 2 num_states: 2


## 6. Map simplexity tokens to project symbols (A, B, C)

For use with the rest of the repo (e.g. prompting an LLM), map integer tokens to symbols. Tensors are PyTorch.

In [48]:
HMM_CONCEPTS = ['A', 'B', 'C']  # same as accuracy_vs_context_HMM.ipynb

def tokens_to_symbols(tokens) -> list:
    """Convert integer token array (PyTorch or numpy) to list of symbols (e.g. for Mess3)."""
    if hasattr(tokens, 'cpu'):
        tokens = tokens.cpu().numpy()
    return [HMM_CONCEPTS[int(t)] for t in tokens]

sample_tokens = inputs[0, :15]  # PyTorch tensor from simplexity output
print("First 15 tokens as symbols:", tokens_to_symbols(sample_tokens))

First 15 tokens as symbols: ['B', 'B', 'B', 'B', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'B', 'B', 'A']


# 7. Plot the belief states

In [ ]:
# from src.hmm.hmm import Mess3HMM, belief_to_barycentric
inputs.shape

torch.Size([20, 999])

In [13]:
belief_states.shape

torch.Size([20, 999, 3])

In [14]:
belief_to_barycentric(belief_states.reshape(-1,3).cpu(), belief_states.reshape(-1,3).cpu())

In [68]:
belief_to_barycentric(belief_states.reshape(-1,3), belief_states.reshape(-1,3))